# 04 — Modeling

**Immunization Defaulter Risk Engine** · Dr. Erick Kiprotich Yegon

Loads the production-trained artifacts and evaluates them in full.
Also demonstrates re-training from scratch for methodological transparency.

**Production model stack:**
- **XGBoost** (gradient-boosted trees, Optuna-tuned hyperparameters)
- **CalibratedClassifierCV** — isotonic regression calibration (5-fold CV)
- **ColumnTransformer** — median imputation (numeric), mode imputation (binary/categorical), ordinal encoding (geographic)
- **Class imbalance** — handled via `scale_pos_weight = n_neg / n_pos`
- **Evaluation** — stratified train/test split (80/20), PR-AUC primary metric

In [5]:
#pip install mlflow

In [6]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import joblib
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR    = PROJECT_ROOT / "reports"

pd.set_option("display.float_format", "{:.4f}".format)

from src.features.pipeline import FeaturePipeline
from src.model.trainer import IZDefaulterTrainer

df = pd.read_parquet(DATA_PROCESSED / "analytical_dataset.parquet")
print(f"Loaded {df.shape[0]:,} patient records")

Loaded 6,864 patient records


## 1. Load production-trained artifacts

In [7]:
model, preprocessor, feature_names = IZDefaulterTrainer.load_artifacts(
    str(DATA_PROCESSED)
)

print("Model type  :", type(model).__name__)
print("Features    :", len(feature_names))
print("\nFeature names:")
for i, f in enumerate(feature_names, 1):
    print(f"  {i:2d}. {f}")

Model type  : CalibratedClassifierCV
Features    : 44

Feature names:
   1. patient_age_in_months
   2. vax_completeness_score
   3. vax_completeness_all
   4. age_expected_vaccine_count
   5. due_count_clean
   6. vitamin_a_completeness
   7. months_since_reported
   8. vax_received_core
   9. vax_received_total
  10. chw_supervision_frequency
  11. chw_immunization_competency_pct
  12. chw_overall_assessment_pct
  13. chw_workload_u2
  14. monthly_homevisit_rate
  15. months_since_last_supervision
  16. patient_sex_binary
  17. is_malaria_endemic_binary
  18. penta_series_complete
  19. opv_series_complete
  20. pcv_series_complete
  21. rota_series_complete
  22. measles_booster_gap
  23. has_bcg
  24. has_opv_0
  25. has_opv_1
  26. has_opv_2
  27. has_opv_3
  28. has_pcv_1
  29. has_pcv_2
  30. has_pcv_3
  31. has_penta_1
  32. has_penta_2
  33. has_penta_3
  34. has_ipv
  35. has_rota_1
  36. has_rota_2
  37. has_rota_3
  38. has_measles_9_months
  39. has_measles_18_months
  40.

In [8]:
# Load last run metrics
with open(DATA_PROCESSED / "run_result.json") as f:
    run_result = json.load(f)

metrics = run_result.get("metrics", {})
print(f"MLflow run ID: {run_result.get('run_id', 'N/A')}")
print()
metrics_df = pd.DataFrame(
    [{"metric": k, "value": round(v, 4)} for k, v in metrics.items()]
)
metrics_df

MLflow run ID: f4f1608564174f038e9cf6cc3ef288d5



,metric,value
0,roc_auc,0.5078
1,pr_auc,0.2464
2,f1,0.0000
3,precision,0.0000
4,recall,0.0000
5,brier_score,0.2149
6,precision_at_20pct,0.0000
7,expected_calibration_error,0.1995
8,roc_auc_male,0.4000
9,roc_auc_female,0.7273


## 2. Re-evaluate on the analytical dataset

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    brier_score_loss, classification_report,
    roc_curve, precision_recall_curve,
    calibration_curve,
)

fp = FeaturePipeline(config_path=str(PROJECT_ROOT / "config" / "model_config.yaml"))
X, y = fp.select_features(df.copy())

# Use same random_state=42 as trainer for comparable split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Score using the production preprocessor + model
X_test_t  = preprocessor.transform(X_test)
y_prob    = model.predict_proba(X_test_t)[:, 1]
y_pred    = (y_prob >= 0.5).astype(int)

eval_metrics = {
    "ROC-AUC"             : roc_auc_score(y_test, y_prob),
    "PR-AUC"              : average_precision_score(y_test, y_prob),
    "F1"                  : f1_score(y_test, y_pred, zero_division=0),
    "Precision"           : precision_score(y_test, y_pred, zero_division=0),
    "Recall"              : recall_score(y_test, y_pred, zero_division=0),
    "Brier Score"         : brier_score_loss(y_test, y_prob),
}

# Precision at top 20%
n_top = max(1, int(0.20 * len(y_test)))
top_idx = np.argsort(y_prob)[-n_top:]
eval_metrics["Precision@Top20%"] = float(y_test.iloc[top_idx].mean())

pd.DataFrame(
    [{"metric": k, "value": round(v, 4)} for k, v in eval_metrics.items()]
)

ImportError: cannot import name 'calibration_curve' from 'sklearn.metrics' (C:\Users\keyeg\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\__init__.py)

In [ ]:
print(classification_report(y_test, y_pred, target_names=["Non-defaulter", "Defaulter"]))

## 3. ROC and Precision-Recall curves

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
prec, rec, thresholds_pr = precision_recall_curve(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC
axes[0].plot(fpr, tpr, color="steelblue", lw=2,
             label=f"XGBoost (AUC={eval_metrics['ROC-AUC']:.3f})")
axes[0].plot([0, 1], [0, 1], "--", color="grey", alpha=0.5, label="Random")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve", fontsize=12, fontweight="bold")
axes[0].legend()
axes[0].fill_between(fpr, tpr, alpha=0.1, color="steelblue")

# PR
axes[1].plot(rec, prec, color="coral", lw=2,
             label=f"XGBoost (PR-AUC={eval_metrics['PR-AUC']:.3f})")
baseline = y_test.mean()
axes[1].axhline(baseline, color="grey", linestyle="--", alpha=0.5,
                label=f"Random ({baseline:.3f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve", fontsize=12, fontweight="bold")
axes[1].legend()
axes[1].fill_between(rec, prec, alpha=0.1, color="coral")

plt.tight_layout()
plt.show()

## 4. Probability calibration curve

In [ ]:
fraction_pos, mean_pred = calibration_curve(y_test, y_prob, n_bins=10, strategy="uniform")

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], "--", color="grey", alpha=0.5, label="Perfect calibration")
ax.plot(mean_pred, fraction_pos, marker="o", color="steelblue", lw=2,
        label="XGBoost (isotonic calibration)")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("Calibration curve", fontsize=12, fontweight="bold")
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

# ECE
bins = np.linspace(0, 1, 11)
ece = 0.0
n = len(y_test)
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (y_prob >= lo) & (y_prob < hi)
    if m.sum() > 0:
        ece += (m.sum() / n) * abs(y_test[m].mean() - y_prob[m].mean())
print(f"Expected Calibration Error (ECE): {ece:.4f}")

## 5. Threshold operating curve

In [ ]:
threshold_table = []
for t in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
    y_t = (y_prob >= t).astype(int)
    flagged = y_t.sum()
    threshold_table.append({
        "threshold"         : t,
        "flagged_n"         : int(flagged),
        "flagged_pct"       : round(flagged / len(y_test) * 100, 1),
        "precision"         : round(precision_score(y_test, y_t, zero_division=0), 3),
        "recall"            : round(recall_score(y_test, y_t, zero_division=0), 3),
        "f1"                : round(f1_score(y_test, y_t, zero_division=0), 3),
    })

thresh_df = pd.DataFrame(threshold_table)
print("Threshold operating table (test set):")
thresh_df

## 6. Fairness — AUC by patient sex

In [ ]:
fairness = []
for sex_val, label in [(1, "Male"), (0, "Female")]:
    mask = X_test["patient_sex_binary"] == sex_val
    if mask.sum() > 10 and y_test[mask].nunique() > 1:
        auc = roc_auc_score(y_test[mask], y_prob[mask])
        prev = y_test[mask].mean()
        fairness.append({
            "sex"       : label,
            "n_patients": int(mask.sum()),
            "prevalence": round(prev, 4),
            "roc_auc"   : round(auc, 4),
        })

fair_df = pd.DataFrame(fairness)
if len(fair_df) == 2:
    gap = abs(fair_df.loc[0, "roc_auc"] - fair_df.loc[1, "roc_auc"])
    print(f"AUC fairness gap (|Male − Female|): {gap:.4f}")
    print("(< 0.05 is generally considered acceptable)")
    print()
fair_df

## 7. Saved report charts

In [ ]:
from IPython.display import display, Image

for label, path in [
    ("ROC / PR Curves",    REPORTS_DIR / "roc_pr_curves.png"),
    ("Calibration Curve",  REPORTS_DIR / "calibration_curve.png"),
    ("Confusion Matrix",   REPORTS_DIR / "confusion_matrix.png"),
    ("Feature Importance", REPORTS_DIR / "feature_importance.png"),
]:
    if Path(path).exists():
        print(f"\n--- {label} ---")
        display(Image(filename=str(path), width=700))
    else:
        print(f"  {label}: not found at {path}")

## 8. Methodological notes

| Decision | Rationale |
|---|---|
| **XGBoost over logistic regression** | Non-linear interactions between age, vaccine history, and CHW context; tree-based models naturally handle missing values |
| **Isotonic calibration** | Raw XGBoost probabilities are poorly calibrated on imbalanced data; isotonic regression brings ECE to near-zero |
| **PR-AUC as tuning objective** | ROC-AUC is optimistic under class imbalance (16.5% positive); PR-AUC directly measures ranking quality for the minority class |
| **scale_pos_weight** | Computed dynamically as `n_neg / n_pos` per training fold — ensures gradient updates reflect actual class imbalance |
| **Median imputation for numeric** | Conservative choice for missing CHW context (some CHW areas have no supervision record); XGBoost is robust to this |
| **Ordinal encoding for geography** | Label encoding + `handle_unknown="use_encoded_value"` for inference on new sub-counties |
| **No temporal validation** | Dataset spans multiple months but no patient ID reappears across months — standard stratified split is appropriate |